In [ ]:
import pandas as pd
import yaml

cfg = yaml.safe_load(open('configs/settings.yaml'))

In [ ]:
def clean_gma_age_sex(cfg, file_path):
    raw = pd.read_excel(file_path, sheet_name='Population', header=None)
    hdr = raw.index[raw[1] == 'Age'][0]          # row with Male/Female/Total labels
    years = raw.iloc[hdr - 1].ffill()            # years row (forward-filled across each group)
    gender = raw.iloc[hdr]
    data = raw.iloc[hdr + 1:]

    parts = []
    for c in range(raw.shape[1]):
        if gender[c] in ('Male', 'Female'):       # skip Total and separator columns
            sub = data[[0, 1, c]].copy()
            sub.columns = ['county', 'age', 'value']
            sub['year'] = int(years[c])
            sub['gender'] = gender[c]
            parts.append(sub)
    long = pd.concat(parts, ignore_index=True)

    long['county_id'] = (long['county'].astype(str).str.strip() + ' County').map(cfg['county_map'])
    long = long.loc[long['county_id'].notna() & (long['age'] != 'Total')]   # drop other counties + all-ages row

    age_order = list(dict.fromkeys(long['age']))
    long['value'] = pd.to_numeric(long['value'], errors='coerce')
    pop = long.groupby(['county_id', 'gender', 'age', 'year'], as_index=False)['value'].sum()
    wide = pop.pivot(index=['county_id', 'gender', 'age'], columns='year', values='value')
    all_years = range(wide.columns.min(), wide.columns.max() + 1)
    wide = wide.reindex(columns=all_years).interpolate(axis=1, method='linear')
    wide = wide.reindex(age_order, level='age').sort_index(level='county_id', sort_remaining=False)
    return wide

gma = clean_gma_age_sex(cfg, 'data/gma_2022_age_sex_med.xlsx')
gma

In [ ]:
gma.to_csv('output/gma_2022_age_sex_med_cleaned.csv')